In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import nltk
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()

In [ ]:
# download if you haven't before
# nltk.download('wordnet')  

In [ ]:
from nltk.corpus import wordnet as wn

def build_embedding_text(synset):
    definition = synset.definition()
    lemmas = ", ".join([str(lemma.name().replace("_", " ")) for lemma in synset.lemmas()])
    embedding = f"""The definition of {synset.name().split('.', 1)[0]} is {definition}. The words in this synset are: {lemmas}."""

    
    if len(synset.hypernyms()) != 0:
        hypernyms = ", ".join([str(hypernym.name()) for hypernym in synset.hypernyms()])
        embedding += f""" Hypernyms of this synset are: {hypernyms}."""
    if len(synset.examples()) != 0:
        examples = ", ".join([str(f"\"{example}\"") for example in synset.examples()])
        embedding += f""" Examples are: {examples}."""
    return embedding

all_embedding_sentences = []
metadata = []
for synset in wn.all_synsets():
    # if len(synset.lemmas()) >= 2:   # add if database building is slow since a lot of synsets seem to have only one word in them
    all_embedding_sentences.append(build_embedding_text(synset))
    metadata.append({
        "synset": synset.name(),
        "definition": synset.definition(),
        "lemmas": [lemma.name() for lemma in synset.lemmas()],
        "hypernyms":[hypernym.name() for hypernym in synset.hypernyms()],
        "pos": synset.pos()
    })

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(all_embedding_sentences, show_progress_bar=True)

In [ ]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

normalized_embeddings = normalize(embeddings)
num_dimensions = embeddings.shape[1]
index = faiss.IndexFlatIP(num_dimensions)
index.add(normalized_embeddings) 

faiss.write_index(index, "wordnet.index")

with open("metadata.pkl", "wb") as f:
    pickle.dump(({"documents": all_embedding_sentences, "metadata": metadata}), f)


# Start here after running above once

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [2]:
# Load FAISS index
index = faiss.read_index("wordnet.index")

# Load documents + metadata
with open("metadata.pkl", "rb") as f:
    data = pickle.load(f)

documents = data["documents"]
metadata = data["metadata"]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

def search(query, k=5):
    
    query_vec = normalize(model.encode([query]))
    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        results.append((documents[i], metadata[i]))

    return results

In [7]:
search("dog, bird")

[('The definition of bird_dog is a gun dog trained to locate or retrieve birds. The words in this synset are: bird dog. Hypernyms of this synset are: sporting_dog.n.01.',
  {'synset': 'bird_dog.n.01',
   'definition': 'a gun dog trained to locate or retrieve birds',
   'lemmas': ['bird_dog'],
   'hypernyms': ['sporting_dog.n.01'],
   'pos': 'n'}),
 ('The definition of bird is the flesh of a bird or fowl (wild or domestic) used as food. The words in this synset are: bird, fowl. Hypernyms of this synset are: meat.n.01.',
  {'synset': 'bird.n.02',
   'definition': 'the flesh of a bird or fowl (wild or domestic) used as food',
   'lemmas': ['bird', 'fowl'],
   'hypernyms': ['meat.n.01'],
   'pos': 'n'}),
 ('The definition of bird is warm-blooded egg-laying vertebrates characterized by feathers and forelimbs modified as wings. The words in this synset are: bird. Hypernyms of this synset are: vertebrate.n.01.',
  {'synset': 'bird.n.01',
   'definition': 'warm-blooded egg-laying vertebrates c

In [ ]:
# Install if needed
# %pip install ipywidgets

import ipywidgets as widgets
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

messages = [
    {"role": "system", "content": "You are a helpful assistant that explains things simply and clearly."}
]

def ask_gpt(prompt):
    messages.append({"role": "user", "content": prompt})
    
    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages
    )
    
    reply = completion.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    
    return reply  


input_box = widgets.Text(
    placeholder="Type your message...",
    description="You:",
    layout=widgets.Layout(width='70%')
)

send_button = widgets.Button(
    description="Send",
    button_style='primary'
)

clear_button = widgets.Button(
    description="Clear Chat",
    button_style='danger'
)

system_box = widgets.Text(
    value=messages[0]["content"],
    description="System:",
    layout=widgets.Layout(width='70%')
)

output_area = widgets.Output()


def send_message(_):
    user_input = input_box.value.strip()
    if not user_input:
        return
    
    input_box.value = ""
    
    with output_area:
        display(Markdown(f"**You:** {user_input}"))
        
        reply = ask_gpt(user_input)
        
        display(Markdown(f"**GPT:** {reply}"))


def clear_chat(_):
    global messages
    messages = [
        {"role": "system", "content": system_box.value}
    ]
    output_area.clear_output()


send_button.on_click(send_message)
input_box.on_submit(send_message)
clear_button.on_click(clear_chat)


display(system_box)
display(widgets.HBox([input_box, send_button, clear_button]))
display(output_area)